# TEST

In [1]:
# Cell 1) imports + run dir 설정
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import re

def pick_latest_run(runs_root: Path) -> Path:
    candidates = [p for p in runs_root.iterdir() if p.is_dir()]
    if not candidates:
        raise FileNotFoundError(f"No run dirs under: {runs_root}")
    return sorted(candidates, key=lambda p: p.name)[-1]

RUN_DIR = Path('..').resolve()

RESULTS_CSV = RUN_DIR / "results.csv"
TRACES_DIR = RUN_DIR / "traces"
harness_report = Path("../exp1-step2-4-qwen-200_1024.json")

print("RUN_DIR =", RUN_DIR)
print("RESULTS_CSV exists?", RESULTS_CSV.exists())
print("TRACES_DIR exists?", TRACES_DIR.exists())

print("run_dir exists:", RUN_DIR.exists(), RUN_DIR)
print("results.csv exists:", RESULTS_CSV.exists())
print("traces dir exists:", TRACES_DIR.exists())
print("harness report exists:", harness_report.exists(), harness_report)

RUN_DIR = /home/dibaeck/workspace/project_IR_sLM_MAS/runs_archive/exp1_qwen2p5_baseline_20260226_101302
RESULTS_CSV exists? True
TRACES_DIR exists? True
run_dir exists: True /home/dibaeck/workspace/project_IR_sLM_MAS/runs_archive/exp1_qwen2p5_baseline_20260226_101302
results.csv exists: True
traces dir exists: True
harness report exists: True ../exp1-step2-4-qwen-200_1024.json


In [2]:
df = pd.read_csv(RESULTS_CSV)
print("Total rows:", len(df))
display(df.head(3))

# 타입 정리(가끔 trial_id가 object로 들어옴)
df["trial_id"] = pd.to_numeric(df["trial_id"], errors="coerce").fillna(0).astype(int)
df["files_changed"] = pd.to_numeric(df.get("files_changed"), errors="coerce")
df["patch_lines_added"] = pd.to_numeric(df.get("patch_lines_added"), errors="coerce")
df["patch_lines_removed"] = pd.to_numeric(df.get("patch_lines_removed"), errors="coerce")

# diff "크기" 지표
df["diff_total_lines"] = (df["patch_lines_added"].fillna(0) + df["patch_lines_removed"].fillna(0)).astype(int)

Total rows: 200


,task_id,trial_id,model,prompt_hash,taxonomy_version,success,stage,error_type,signature,returncode,...,format_used,format_ok,format_reason,apply_check_ok,apply_check_reason,patch_lines_added,patch_lines_removed,files_changed,timestamp,seed
0,astropy__astropy-12907,0,Qwen/Qwen2.5-Coder-7B-Instruct,1d006f36813d2fc095bc8da160e7e87abfd8d05ee8957d...,step2-4,False,EXEC,EXEC_FAIL,docker_image_not_found,125.0,...,NaN,NaN,NaN,NaN,NaN,4,1,1,2026-02-26T10:13:02.250924,42
1,astropy__astropy-14182,0,Qwen/Qwen2.5-Coder-7B-Instruct,323a3b727d26875c53e665ce8dcd10f792793ee5799a7a...,step2-4,False,EDIT_PARSE,GEN_FAIL,invalid_edit_script,-1.0,...,NaN,NaN,NaN,NaN,NaN,0,0,0,2026-02-26T10:13:07.087934,42
2,astropy__astropy-14365,0,Qwen/Qwen2.5-Coder-7B-Instruct,232a841813d672f784b09b1e96ec3c80e68dec2004a896...,step2-4,False,EXEC,EXEC_FAIL,docker_image_not_found,125.0,...,NaN,NaN,NaN,NaN,NaN,7,1,1,2026-02-26T10:13:17.954124,42


In [3]:
traces_dir = TRACES_DIR

def safe_read(p: Path):
    try:
        return p.read_text(encoding="utf-8", errors="ignore")
    except Exception:
        return None

# trace_path 만들기: {task_id}_trial{trial}.json
def trace_json_path(task_id, trial_id):
    return traces_dir / f"{task_id}_trial{trial_id}.json"

def edit_json_path(task_id, trial_id):
    return traces_dir / f"{task_id}_trial{trial_id}.edit.json"

def diff_path(task_id, trial_id):
    return traces_dir / f"{task_id}_trial{trial_id}.patch.diff"

# traces에서 필요한 필드만 추출
rows = []
for _, r in df.iterrows():
    tid = r["task_id"]
    tr = r["trial_id"]
    tj = trace_json_path(tid, tr)
    ej = edit_json_path(tid, tr)
    dp = diff_path(tid, tr)

    trace_txt = safe_read(tj)
    edit_txt = safe_read(ej)
    diff_txt = safe_read(dp)

    stderr_trace = None
    stdout_trace = None
    issue_text = None
    test_command = None

    if trace_txt:
        try:
            obj = json.loads(trace_txt)
            stderr_trace = obj.get("stderr")
            stdout_trace = obj.get("stdout")
            issue_text = obj.get("issue_text")
            test_command = obj.get("test_command")
        except Exception:
            pass

    rows.append({
        "task_id": tid,
        "trial_id": tr,
        "trace_path": str(tj) if tj.exists() else None,
        "stderr_trace": stderr_trace,
        "stdout_trace": stdout_trace,
        "issue_text": issue_text,
        "test_command_trace": test_command,
        "edit_script_trace": edit_txt,
        "diff_trace": diff_txt,
    })

df_tr = pd.DataFrame(rows)
df_join = df.merge(df_tr, on=["task_id","trial_id"], how="left")

print("Joined rows:", len(df_join))
display(df_join.head(3))

Joined rows: 200


,task_id,trial_id,model,prompt_hash,taxonomy_version,success,stage,error_type,signature,returncode,...,timestamp,seed,diff_total_lines,trace_path,stderr_trace,stdout_trace,issue_text,test_command_trace,edit_script_trace,diff_trace
0,astropy__astropy-12907,0,Qwen/Qwen2.5-Coder-7B-Instruct,1d006f36813d2fc095bc8da160e7e87abfd8d05ee8957d...,step2-4,False,EXEC,EXEC_FAIL,docker_image_not_found,125.0,...,2026-02-26T10:13:02.250924,42,5,/home/dibaeck/workspace/project_IR_sLM_MAS/run...,Unable to find image 'swebench/sweb.eval.x86_6...,,None,echo 'No test command',"{\n ""edits"": [\n {\n ""op""...",diff --git a/astropy/modeling/core.py b/astrop...
1,astropy__astropy-14182,0,Qwen/Qwen2.5-Coder-7B-Instruct,323a3b727d26875c53e665ce8dcd10f792793ee5799a7a...,step2-4,False,EDIT_PARSE,GEN_FAIL,invalid_edit_script,-1.0,...,2026-02-26T10:13:07.087934,42,0,/home/dibaeck/workspace/project_IR_sLM_MAS/run...,"Expecting ',' delimiter: line 8 column 6 (char...",,None,,"{\n ""edits"": [\n {\n ""op"": ""insert_af...",None
2,astropy__astropy-14365,0,Qwen/Qwen2.5-Coder-7B-Instruct,232a841813d672f784b09b1e96ec3c80e68dec2004a896...,step2-4,False,EXEC,EXEC_FAIL,docker_image_not_found,125.0,...,2026-02-26T10:13:17.954124,42,8,/home/dibaeck/workspace/project_IR_sLM_MAS/run...,Unable to find image 'swebench/sweb.eval.x86_6...,,None,echo 'No test command',"{\n ""edits"": [\n {\n ""op"": ""replace_r...",diff --git a/astropy/io/ascii/qdp.py b/astropy...


In [4]:
with open(harness_report, "r", encoding="utf-8") as f:
    rep = json.load(f)

# harness report는 schema_version 2에서 resolved_ids / unresolved_ids / error_ids 같은 리스트가 있음
resolved = set(rep.get("resolved_ids", []) or [])
unresolved = set(rep.get("unresolved_ids", []) or [])
error_ids = set(rep.get("error_ids", []) or [])
submitted = set(rep.get("submitted_ids", []) or [])

# not_submitted = dataset 전체 - submitted
# dataset 전체 id 목록은 report에 없을 수 있으니, 여기서는 df_join 기준으로만 표시
def harness_status(iid: str):
    if iid in resolved: return "resolved"
    if iid in error_ids: return "error"
    if iid in unresolved: return "unresolved"
    if iid in submitted: return "submitted_unknown"
    return "not_submitted"

df_join["harness_status"] = df_join["task_id"].map(harness_status)

display(df_join["harness_status"].value_counts().to_frame("count"))

,count
harness_status,
unresolved,153
not_submitted,46
resolved,1


### edit op 유형 파싱( insert_after / replace_range )

In [5]:
def parse_ops(edit_script_text: str):
    """
    returns (ok, ops_list)
    """
    if not isinstance(edit_script_text, str) or not edit_script_text.strip():
        return False, []
    try:
        obj = json.loads(edit_script_text)
        edits = obj.get("edits", [])
        if not isinstance(edits, list):
            return False, []
        ops = []
        for e in edits:
            if isinstance(e, dict) and "op" in e:
                ops.append(str(e["op"]))
        return True, ops
    except Exception:
        return False, []

ops_ok = []
ops_list = []
n_edits = []
for s in df_join["edit_script_trace"].tolist():
    ok, ops = parse_ops(s)
    ops_ok.append(ok)
    ops_list.append(ops)
    n_edits.append(len(ops))

df_join["ops_parse_ok"] = ops_ok
df_join["ops_list"] = ops_list
df_join["n_edits"] = n_edits

# op one-hot(빈도 분석용)
all_ops = sorted({op for ops in ops_list for op in ops})
for op in all_ops:
    df_join[f"op__{op}"] = df_join["ops_list"].apply(lambda xs: int(op in xs))

print("Detected ops:", all_ops)
display(df_join[["task_id","harness_status","n_edits","ops_list"]].head(5))

Detected ops: ['insert_after', 'replace_range']


,task_id,harness_status,n_edits,ops_list
0,astropy__astropy-12907,unresolved,1,[insert_after]
1,astropy__astropy-14182,not_submitted,0,[]
2,astropy__astropy-14365,unresolved,2,"[replace_range, insert_after]"
3,astropy__astropy-14995,unresolved,1,[replace_range]
4,astropy__astropy-6938,not_submitted,1,[replace_range]


In [6]:
target = df_join[df_join["harness_status"].isin(["resolved","unresolved"])].copy()
print("resolved:", (target["harness_status"]=="resolved").sum(),
      "unresolved:", (target["harness_status"]=="unresolved").sum())

num_cols = [
    "diff_total_lines",
    "patch_lines_added","patch_lines_removed","files_changed",
    "context_num_files",
    "n_edits",
]

# 없는 컬럼 대비
num_cols = [c for c in num_cols if c in target.columns]

summary = target.groupby("harness_status")[num_cols].agg(["count","mean","median","std","min","max"])
display(summary)

resolved: 1 unresolved: 153


diff_total_lines                                     \
                          count      mean median       std min max   
harness_status                                                       
resolved                      1  6.000000    6.0       NaN   6   6   
unresolved                  153  6.797386    5.0  5.166134   1  32   

               patch_lines_added                             ...  \
                           count      mean median       std  ...   
harness_status                                               ...   
resolved                       1  2.000000    2.0       NaN  ...   
unresolved                   153  4.836601    3.0  5.185325  ...   

               context_num_files                   n_edits                   \
                          median       std min max   count      mean median   
harness_status                                                                
resolved                    80.0       NaN  80  80       1  1.000000    1.0   
unresolved                  80.0  0.515998  75  80     153  1.509804    1.0   

                                  
                     std min max  
harness_status                    
resolved             NaN   1   1  
unresolved      0.795693   1   8  

[2 rows x 36 columns]

In [7]:
def describe_diff(col):
    a = target[target["harness_status"]=="resolved"][col].dropna()
    b = target[target["harness_status"]=="unresolved"][col].dropna()
    if len(a)==0 or len(b)==0:
        return None
    return {
        "col": col,
        "resolved_value(s)": a.tolist(),
        "unresolved_mean": float(b.mean()),
        "unresolved_median": float(b.median()),
        "unresolved_std": float(b.std()) if len(b)>1 else 0.0
    }

rows = []
for c in num_cols:
    d = describe_diff(c)
    if d: rows.append(d)

display(pd.DataFrame(rows))

,col,resolved_value(s),unresolved_mean,unresolved_median,unresolved_std
0,diff_total_lines,[6],6.797386,5.0,5.166134
1,patch_lines_added,[2],4.836601,3.0,5.185325
2,patch_lines_removed,[4],1.960784,2.0,1.394927
3,files_changed,[1],1.013072,1.0,0.113956
4,context_num_files,[80],79.941176,80.0,0.515998
5,n_edits,[1],1.509804,1.0,0.795693


## Resolved 살펴보기

In [8]:
sol = df_join[df_join["harness_status"]=="resolved"].copy()
print("resolved rows:", len(sol))
if len(sol):
    r = sol.iloc[0]
    print("TASK:", r["task_id"], "trial:", r["trial_id"])
    print("\n== ops_list ==")
    print(r.get("ops_list"))
    print("\n== n_edits ==")
    print(r.get("n_edits"))
    print("\n== diff stats ==")
    print("added:", r.get("patch_lines_added"), "removed:", r.get("patch_lines_removed"), "files_changed:", r.get("files_changed"))
    print("\n== diff preview ==")
    print((r.get("diff_trace") or "")[:2000])
    print("\n== stderr preview ==")
    print((r.get("stderr_trace") or "")[:1000])

resolved rows: 1
TASK: psf__requests-863 trial: 0

== ops_list ==
['replace_range']

== n_edits ==
1

== diff stats ==
added: 2 removed: 4 files_changed: 1

== diff preview ==
diff --git a/requests/hooks.py b/requests/hooks.py
index 9e0ce346..0ad82b92 100644
--- a/requests/hooks.py
+++ b/requests/hooks.py
@@ -17,10 +17,8 @@ Available hooks:
 ``pre_send``:
     The Request object, directly before being sent.
 
-``post_request``:
-    The Request object, directly after being sent.
-
-``response``:
+        for name, hooks in hooks.items():
+            self._hooks[name] = [hook for hook in hooks if callable(hook)]``response``:
     The response generated from a Request.
 
 """


== stderr preview ==
Unable to find image 'swebench/sweb.eval.x86_64:latest' locally
docker: Error response from daemon: pull access denied for swebench/sweb.eval.x86_64, repository does not exist or may require 'docker login'

Run 'docker run --help' for more information



### Unresolved 살펴보기

- signature/stage : 이 변수는 내가 Executor로 설정한 값의 결과(내 루프에서의 결과물)    

> Harness 결과에서는   
> - error_instances = 0 
> - completed_instances = 154    
> 하네스는 Docker 문제 없이 정상 평가함   

-> 즉, "docker_image_not_found"는 내가 잘못필터링함

- diff_total_lines : 수정 줄
- files_changed : 수정한 파일 수
- n_edits :
- ops_list :    
   
작은 diff(1줄)와 큰 diff(30줄)가 모두 실패

In [9]:
uns = df_join[df_join["harness_status"]=="unresolved"].copy()
if len(uns):
    display(uns.sort_values("diff_total_lines", ascending=False)[["task_id","diff_total_lines","files_changed","n_edits","ops_list","signature","stage"]].head(5))
    display(uns.sort_values("diff_total_lines", ascending=True)[["task_id","diff_total_lines","files_changed","n_edits","ops_list","signature","stage"]].head(5))

,task_id,diff_total_lines,files_changed,n_edits,ops_list,signature,stage
46,django__django-13028,32,1,1,[replace_range],docker_image_not_found,EXEC
107,django__django-16229,31,1,1,[insert_after],docker_image_not_found,EXEC
69,django__django-14155,21,1,3,"[replace_range, insert_after, replace_range]",docker_image_not_found,EXEC
48,django__django-13158,21,1,2,"[replace_range, insert_after]",docker_image_not_found,EXEC
66,django__django-13964,21,2,2,"[insert_after, insert_after]",docker_image_not_found,EXEC


,task_id,diff_total_lines,files_changed,n_edits,ops_list,signature,stage
74,django__django-14580,1,1,1,[insert_after],docker_image_not_found,EXEC
153,psf__requests-2674,1,1,1,[insert_after],docker_image_not_found,EXEC
89,django__django-15320,1,1,1,[insert_after],docker_image_not_found,EXEC
85,django__django-15061,1,1,1,[replace_range],docker_image_not_found,EXEC
128,matplotlib__matplotlib-23913,1,1,1,[insert_after],docker_image_not_found,EXEC


---

- diff 크기 차이
- 수정 파일 수 차이
- context 사용 차이
- edit op 유형 차이

In [ ]:
import json, re
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_colwidth", 200)
pd.set_option("display.max_columns", 200)

In [11]:
df = pd.read_csv(RESULTS_CSV)

# 타입 정리(있을 수도/없을 수도 있는 컬럼 대비)
for c in ["success", "context_used", "edit_used", "edit_parse_ok", "diff_export_ok"]:
    if c in df.columns:
        df[c] = df[c].astype("boolean")

# 식별용
if "task_id" not in df.columns:
    raise ValueError("results.csv에 task_id 컬럼이 없음")

df["trial_id"] = df["trial_id"].fillna(0).astype(int)

print("Total rows:", len(df))
display(df.head(3))

Total rows: 200


,task_id,trial_id,model,prompt_hash,taxonomy_version,success,stage,error_type,signature,returncode,...,format_used,format_ok,format_reason,apply_check_ok,apply_check_reason,patch_lines_added,patch_lines_removed,files_changed,timestamp,seed
0,astropy__astropy-12907,0,Qwen/Qwen2.5-Coder-7B-Instruct,1d006f36813d2fc095bc8da160e7e87abfd8d05ee8957d...,step2-4,False,EXEC,EXEC_FAIL,docker_image_not_found,125.0,...,NaN,NaN,NaN,NaN,NaN,4,1,1,2026-02-26T10:13:02.250924,42
1,astropy__astropy-14182,0,Qwen/Qwen2.5-Coder-7B-Instruct,323a3b727d26875c53e665ce8dcd10f792793ee5799a7a...,step2-4,False,EDIT_PARSE,GEN_FAIL,invalid_edit_script,-1.0,...,NaN,NaN,NaN,NaN,NaN,0,0,0,2026-02-26T10:13:07.087934,42
2,astropy__astropy-14365,0,Qwen/Qwen2.5-Coder-7B-Instruct,232a841813d672f784b09b1e96ec3c80e68dec2004a896...,step2-4,False,EXEC,EXEC_FAIL,docker_image_not_found,125.0,...,NaN,NaN,NaN,NaN,NaN,7,1,1,2026-02-26T10:13:17.954124,42


In [13]:
def parse_edit_ops(edit_script: str):
    """
    edit_script JSON 문자열에서 edits op 리스트 추출.
    실패하면 빈 리스트 반환.
    """
    if not isinstance(edit_script, str) or not edit_script.strip():
        return []
    try:
        data = json.loads(edit_script)
        edits = data.get("edits", [])
        if not isinstance(edits, list):
            return []
        ops = []
        for e in edits:
            if isinstance(e, dict):
                ops.append(str(e.get("op", "")).strip())
        return [o for o in ops if o]
    except Exception:
        return []

def diff_total_lines(diff_text: str) -> int:
    if not isinstance(diff_text, str) or not diff_text:
        return 0
    return len(diff_text.splitlines())

def load_traces(traces_dir: Path) -> pd.DataFrame:
    rows = []
    for p in sorted(traces_dir.glob("*.json")):
        try:
            obj = json.loads(p.read_text(encoding="utf-8", errors="ignore"))
        except Exception:
            continue

        task_id  = obj.get("task_id")
        trial_id = obj.get("trial_id", 0)

        # trace_data에 들어있는 키들 기준
        stderr = obj.get("stderr") or ""
        stdout = obj.get("stdout") or ""
        diff   = obj.get("diff") or ""
        edit_script = obj.get("edit_script") or ""

        ops = parse_edit_ops(edit_script)

        rows.append({
            "task_id": task_id,
            "trial_id": int(trial_id) if str(trial_id).isdigit() else 0,
            "trace_path": str(p),
            "stderr_trace": stderr,
            "stdout_trace": stdout,
            "diff_trace": diff,
            "edit_script_trace": edit_script,
            "diff_total_lines_trace": diff_total_lines(diff),
            "ops_list": ops,
            "n_edits": len(ops),
        })
    return pd.DataFrame(rows)

df_tr = load_traces(TRACES_DIR)
print("Traces loaded:", len(df_tr))
display(df_tr.head(3))

Traces loaded: 389


,task_id,trial_id,trace_path,stderr_trace,stdout_trace,diff_trace,edit_script_trace,diff_total_lines_trace,ops_list,n_edits
0,None,0,/home/dibaeck/workspace/project_IR_sLM_MAS/run...,,,,,0,[],0
1,astropy__astropy-12907,0,/home/dibaeck/workspace/project_IR_sLM_MAS/run...,Unable to find image 'swebench/sweb.eval.x86_6...,,diff --git a/astropy/modeling/core.py b/astrop...,"{\n ""edits"": [\n {\n ""op""...",16,[insert_after],1
2,astropy__astropy-14182,0,/home/dibaeck/workspace/project_IR_sLM_MAS/run...,"Expecting ',' delimiter: line 8 column 6 (char...",,,"{\n ""edits"": [\n {\n ""op"": ""insert_af...",0,[],0


In [14]:
df_join = df.merge(df_tr, on=["task_id", "trial_id"], how="left")

print("Joined rows:", len(df_join))
print("Missing trace rows:", df_join["trace_path"].isna().sum())
display(df_join.head(3))

Joined rows: 200
Missing trace rows: 0


,task_id,trial_id,model,prompt_hash,taxonomy_version,success,stage,error_type,signature,returncode,...,timestamp,seed,trace_path,stderr_trace,stdout_trace,diff_trace,edit_script_trace,diff_total_lines_trace,ops_list,n_edits
0,astropy__astropy-12907,0,Qwen/Qwen2.5-Coder-7B-Instruct,1d006f36813d2fc095bc8da160e7e87abfd8d05ee8957d...,step2-4,False,EXEC,EXEC_FAIL,docker_image_not_found,125.0,...,2026-02-26T10:13:02.250924,42,/home/dibaeck/workspace/project_IR_sLM_MAS/run...,Unable to find image 'swebench/sweb.eval.x86_6...,,diff --git a/astropy/modeling/core.py b/astrop...,"{\n ""edits"": [\n {\n ""op""...",16,[insert_after],1
1,astropy__astropy-14182,0,Qwen/Qwen2.5-Coder-7B-Instruct,323a3b727d26875c53e665ce8dcd10f792793ee5799a7a...,step2-4,False,EDIT_PARSE,GEN_FAIL,invalid_edit_script,-1.0,...,2026-02-26T10:13:07.087934,42,/home/dibaeck/workspace/project_IR_sLM_MAS/run...,"Expecting ',' delimiter: line 8 column 6 (char...",,,"{\n ""edits"": [\n {\n ""op"": ""insert_af...",0,[],0
2,astropy__astropy-14365,0,Qwen/Qwen2.5-Coder-7B-Instruct,232a841813d672f784b09b1e96ec3c80e68dec2004a896...,step2-4,False,EXEC,EXEC_FAIL,docker_image_not_found,125.0,...,2026-02-26T10:13:17.954124,42,/home/dibaeck/workspace/project_IR_sLM_MAS/run...,Unable to find image 'swebench/sweb.eval.x86_6...,,diff --git a/astropy/io/ascii/qdp.py b/astropy...,"{\n ""edits"": [\n {\n ""op"": ""replace_r...",16,"[replace_range, insert_after]",2


In [15]:
rep = json.loads(harness_report.read_text(encoding="utf-8"))

resolved_ids   = set(rep.get("resolved_ids", []))
unresolved_ids = set(rep.get("unresolved_ids", []))
error_ids      = set(rep.get("error_ids", []))
submitted_ids  = set(rep.get("submitted_ids", []))

def harness_status(iid: str):
    if iid in error_ids:
        return "error"
    if iid in resolved_ids:
        return "resolved"
    if iid in unresolved_ids:
        return "unresolved"
    if iid in submitted_ids:
        # submitted인데 resolved/unresolved/error에 없으면 스키마/부분완료 케이스 가능
        return "submitted_unknown"
    return "not_submitted"

df_join["harness_status"] = df_join["task_id"].map(harness_status)

print(df_join["harness_status"].value_counts(dropna=False))

harness_status
unresolved       153
not_submitted     46
resolved           1
Name: count, dtype: int64


In [16]:
resolved = df_join[df_join["harness_status"] == "resolved"].copy()
unres    = df_join[df_join["harness_status"] == "unresolved"].copy()
errs     = df_join[df_join["harness_status"] == "error"].copy()

print("resolved:", len(resolved))
print("unresolved:", len(unres))
print("error:", len(errs))

cols = [
    "diff_total_lines_trace",  # traces에서 계산한 diff 총 라인
    "files_changed",           # results.csv에 있으면
    "context_used",
    "context_num_files",
    "n_edits",
]
cols = [c for c in cols if c in df_join.columns]

summary = pd.concat([
    resolved[cols].assign(group="resolved"),
    unres[cols].assign(group="unresolved"),
], ignore_index=True).groupby("group").agg(["count","mean","median","min","max"])

display(summary)

resolved: 1
unresolved: 153
error: 0


diff_total_lines_trace                           files_changed  \
                            count       mean median min max         count   
group                                                                       
resolved                        1  17.000000   17.0  17  17             1   
unresolved                    153  18.686275   17.0  10  42           153   

                                     ... context_num_files                    \
                mean median min max  ...             count       mean median   
group                                ...                                       
resolved    1.000000    1.0   1   1  ...                 1  80.000000   80.0   
unresolved  1.013072    1.0   1   2  ...               153  79.941176   80.0   

                   n_edits                           
           min max   count      mean median min max  
group                                                
resolved    80  80       1  1.000000    1.0   1   1  
unresolved  75  80     153  1.509804    1.0   1   8  

[2 rows x 25 columns]

In [17]:
# 1) diff_total_lines 기준 비교
if "diff_total_lines_trace" in df_join.columns:
    display(
        pd.DataFrame({
            "resolved_diff_total_lines": [resolved["diff_total_lines_trace"].tolist()],
            "unresolved_diff_total_lines_summary": [unres["diff_total_lines_trace"].describe().to_dict()]
        })
    )

# 2) files_changed 기준 비교
if "files_changed" in df_join.columns:
    display(
        pd.DataFrame({
            "resolved_files_changed": [resolved["files_changed"].tolist()],
            "unresolved_files_changed_summary": [unres["files_changed"].describe().to_dict()]
        })
    )

# unresolved에서 diff 큰/작은 샘플 5개씩
if "diff_total_lines_trace" in df_join.columns:
    display(unres.sort_values("diff_total_lines_trace", ascending=False)[
        ["task_id","trial_id","diff_total_lines_trace","files_changed","n_edits","ops_list","stage","error_type","signature"]
    ].head(5))
    display(unres.sort_values("diff_total_lines_trace", ascending=True)[
        ["task_id","trial_id","diff_total_lines_trace","files_changed","n_edits","ops_list","stage","error_type","signature"]
    ].head(5))

,resolved_diff_total_lines,unresolved_diff_total_lines_summary
0,[17],"{'count': 153.0, 'mean': 18.686274509803923, '..."


,resolved_files_changed,unresolved_files_changed_summary
0,[1],"{'count': 153.0, 'mean': 1.0130718954248366, '..."


,task_id,trial_id,diff_total_lines_trace,files_changed,n_edits,ops_list,stage,error_type,signature
107,django__django-16229,0,42,1,1,[insert_after],EXEC,EXEC_FAIL,docker_image_not_found
46,django__django-13028,0,40,1,1,[replace_range],EXEC,EXEC_FAIL,docker_image_not_found
66,django__django-13964,0,39,2,2,"[insert_after, insert_after]",EXEC,EXEC_FAIL,docker_image_not_found
69,django__django-14155,0,34,1,3,"[replace_range, insert_after, replace_range]",EXEC,EXEC_FAIL,docker_image_not_found
169,pytest-dev__pytest-5103,0,34,1,2,"[insert_after, replace_range]",EXEC,EXEC_FAIL,docker_image_not_found


,task_id,trial_id,diff_total_lines_trace,files_changed,n_edits,ops_list,stage,error_type,signature
74,django__django-14580,0,10,1,1,[insert_after],EXEC,EXEC_FAIL,docker_image_not_found
168,pytest-dev__pytest-11148,0,10,1,1,[replace_range],EXEC,EXEC_FAIL,docker_image_not_found
171,pytest-dev__pytest-5227,0,11,1,1,[replace_range],EXEC,EXEC_FAIL,docker_image_not_found
31,django__django-12184,0,12,1,2,"[replace_range, insert_after]",EXEC,EXEC_FAIL,docker_image_not_found
126,matplotlib__matplotlib-23562,0,12,1,2,"[replace_range, insert_after]",EXEC,EXEC_FAIL,docker_image_not_found


In [18]:
if "context_used" in df_join.columns:
    tab = pd.crosstab(df_join["harness_status"], df_join["context_used"], dropna=False)
    display(tab)

if "context_num_files" in df_join.columns:
    display(
        pd.concat([
            resolved[["context_num_files"]].assign(group="resolved"),
            unres[["context_num_files"]].assign(group="unresolved"),
        ]).groupby("group")["context_num_files"].describe()
    )

context_used,True
harness_status,
not_submitted,46
resolved,1
unresolved,153


,count,mean,std,min,25%,50%,75%,max
group,,,,,,,,
resolved,1.0,80.000000,NaN,80.0,80.0,80.0,80.0,80.0
unresolved,153.0,79.941176,0.515998,75.0,80.0,80.0,80.0,80.0


In [19]:
if len(resolved) == 1:
    r = resolved.iloc[0]
    r_diff = r.get("diff_total_lines_trace", 0)
    r_files = r.get("files_changed", np.nan)
    r_nedits = r.get("n_edits", 0)

    cand = unres.copy()
    if "diff_total_lines_trace" in cand.columns:
        cand["abs_diff_gap"] = (cand["diff_total_lines_trace"] - r_diff).abs()
    else:
        cand["abs_diff_gap"] = np.nan

    if "files_changed" in cand.columns and not pd.isna(r_files):
        cand["abs_files_gap"] = (cand["files_changed"] - r_files).abs()
    else:
        cand["abs_files_gap"] = 0

    cand["abs_nedits_gap"] = (cand["n_edits"] - r_nedits).abs()

    cand = cand.sort_values(["abs_diff_gap","abs_files_gap","abs_nedits_gap"]).head(10)

    print("Resolved instance:", r["task_id"])
    display(pd.DataFrame([{
        "task_id": r["task_id"],
        "diff_total_lines": r.get("diff_total_lines_trace"),
        "files_changed": r.get("files_changed"),
        "n_edits": r.get("n_edits"),
        "ops_list": r.get("ops_list"),
        "stage": r.get("stage"),
        "signature": r.get("signature"),
    }]))
    print("Closest 10 unresolved:")
    display(cand[["task_id","trial_id","diff_total_lines_trace","files_changed","n_edits","ops_list","stage","signature","abs_diff_gap","abs_files_gap","abs_nedits_gap"]])
else:
    print("resolved가 1개가 아니어서(혹은 0개) 이 셀은 스킵")

Resolved instance: psf__requests-863


,task_id,diff_total_lines,files_changed,n_edits,ops_list,stage,signature
0,psf__requests-863,17,1,1,[replace_range],EXEC,docker_image_not_found


Closest 10 unresolved:


,task_id,trial_id,diff_total_lines_trace,files_changed,n_edits,ops_list,stage,signature,abs_diff_gap,abs_files_gap,abs_nedits_gap
5,astropy__astropy-7746,0,17,1,1,[replace_range],EXEC,docker_image_not_found,0,0,0
9,django__django-11019,0,17,1,1,[replace_range],EXEC,docker_image_not_found,0,0,0
32,django__django-12284,0,17,1,1,[replace_range],EXEC,docker_image_not_found,0,0,0
52,django__django-13315,0,17,1,1,[replace_range],EXEC,docker_image_not_found,0,0,0
59,django__django-13658,0,17,1,1,[replace_range],EXEC,docker_image_not_found,0,0,0
62,django__django-13757,0,17,1,1,[replace_range],EXEC,docker_image_not_found,0,0,0
84,django__django-14999,0,17,1,1,[replace_range],EXEC,docker_image_not_found,0,0,0
131,matplotlib__matplotlib-24149,0,17,1,1,[replace_range],EXEC,docker_image_not_found,0,0,0
11,django__django-11049,0,17,1,2,"[replace_range, insert_after]",EXEC,docker_image_not_found,0,0,1
30,django__django-12125,0,17,1,2,"[replace_range, insert_after]",EXEC,docker_image_not_found,0,0,1


In [20]:
def show_case(df_row):
    print("="*100)
    print("task_id:", df_row["task_id"])
    print("harness_status:", df_row["harness_status"])
    print("stage/signature:", df_row.get("stage"), "/", df_row.get("signature"))
    print("diff_total_lines:", df_row.get("diff_total_lines_trace"))
    print("files_changed:", df_row.get("files_changed"))
    print("n_edits / ops:", df_row.get("n_edits"), "/", df_row.get("ops_list"))
    print("\n--- DIFF (head 60 lines) ---")
    diff = df_row.get("diff_trace") or ""
    print("\n".join(diff.splitlines()[:60]))

if len(resolved)==1:
    show_case(resolved.iloc[0])

    # unresolved에서 diff_total_lines가 가장 가까운 3개
    if "diff_total_lines_trace" in unres.columns:
        rdiff = resolved.iloc[0].get("diff_total_lines_trace", 0)
        top3 = unres.assign(gap=(unres["diff_total_lines_trace"]-rdiff).abs()).sort_values("gap").head(3)
        for _, row in top3.iterrows():
            show_case(row)
            

task_id: psf__requests-863
harness_status: resolved
stage/signature: EXEC / docker_image_not_found
diff_total_lines: 17
files_changed: 1
n_edits / ops: 1 / ['replace_range']

--- DIFF (head 60 lines) ---
diff --git a/requests/hooks.py b/requests/hooks.py
index 9e0ce346..0ad82b92 100644
--- a/requests/hooks.py
+++ b/requests/hooks.py
@@ -17,10 +17,8 @@ Available hooks:
 ``pre_send``:
     The Request object, directly before being sent.
 
-``post_request``:
-    The Request object, directly after being sent.
-
-``response``:
+        for name, hooks in hooks.items():
+            self._hooks[name] = [hook for hook in hooks if callable(hook)]``response``:
     The response generated from a Request.
 
 """
task_id: astropy__astropy-7746
harness_status: unresolved
stage/signature: EXEC / docker_image_not_found
diff_total_lines: 17
files_changed: 1
n_edits / ops: 1 / ['replace_range']

--- DIFF (head 60 lines) ---
diff --git a/astropy/wcs/wcs.py b/astropy/wcs/wcs.py
index d3a06f2b31..7cac932